# baseline — Qwen3-VL + LoRA + Label Smoothing

## 한눈에 보기

1. 이미지는 **원본 종횡비를 유지**한 채 processor 가 픽셀 예산(`MIN_PIXELS`~`MAX_PIXELS`) 안에서 **동적으로** 리사이즈합니다. 별도 전처리(리사이즈 저장)는 없습니다.
2. Qwen3-VL에 `이미지 + 질문 + 4개 보기`를 입력합니다.
3. Attention 중심 LoRA로 1 epoch 학습합니다.
4. loss는 **정답 글자 토큰만** 계산하고 `label_smoothing=0.1`을 적용합니다.
5. 검증은 **`data/split/validation.csv`**(train의 10%, `eda/split_validation.py`가 만든 고정 분할)로 하고, 검증 정확도가 가장 높은 LoRA를 저장합니다.
6. 추론은 생성하지 않고 `a/b/c/d` 확률을 직접 비교합니다.

## 왜 loss를 바꾸나?

관측한 train 재추론 결과에서 6,714개 중 **70.7%가 p≥0.99**였고, 오답 112개 중 **11개는 p≥0.9로 틀렸습니다**.
CE는 정답 확률을 계속 1에 가깝게 밀 수 있으므로, 작은 데이터에서는 과확신/암기 신호가 커질 수 있습니다.

이번 baseline에서는 가장 단순하고 안전한 변경만 넣습니다.

- 기존: assistant 응답의 `정답 글자 + 종료 토큰 + 줄바꿈`에 CE
- 변경: **정답 글자 1개만 loss 대상으로 유지**
- 변경: `label_smoothing_factor=0.1`
- 보류: 4지선다 전용 custom loss, DPO는 별도 실험으로 분리

즉, 먼저 **“정답 한 글자를 덜 과확신하게 학습”**하는 효과만 확인합니다.


> 실행 환경: RunPod pod의 Jupyter(`uv run scripts/runpod.py up`, README 참고). 데이터·모델 캐시·결과는 `/workspace`(Network Volume)에, 압축 푼 데이터·분할 CSV와 venv는 컨테이너 디스크에 둡니다. 로컬 GPU에서도 그대로 돌아갑니다(경로만 repo의 `data/`, `output/`).
>
> 성능/속도 설정: batch 8, grad accumulation 1, dataloader worker 4, eval 300 step, max sequence 2048, BF16+TF32.


# 0. 데이터 압축 해제 + train/validation 분할

- `DATA_ZIP`(RunPod: `/workspace/stv/data.zip`, 로컬: repo의 `data.zip`)을 `DATA_ROOT/raw`에 풉니다.
  RunPod에서는 `DATA_ROOT=/root/data`(컨테이너 디스크)라서 이미지 로딩이 빠른 대신 pod를 지우면 사라지고, 다음 `up` 때 이 셀이 다시 풉니다(1~2분).
- 이미 `train.csv`가 존재하면 압축 해제를 건너뜁니다.
- `DATA_ROOT/split`에 `train.csv`(90%) / `validation.csv`(10%)가 **없으면** `eda/split_validation.py`로 만들고, **있으면 건너뜁니다**(약 30초).
  분할은 이미지 해시 그룹 단위 + 정답 층화, seed 42 고정이라 누가 만들어도 같습니다. 검증 id 목록은 `eda/tables/validation_ids.csv`에 버전 관리됩니다.
- 이미지 리사이즈 전처리는 하지 않습니다. 원본 이미지를 그대로 읽고 모델 processor 가 종횡비를 유지한 채 동적으로 크기를 맞춥니다.


In [ ]:
from pathlib import Path
import os, sys, zipfile

ON_RUNPOD = bool(os.environ.get("RUNPOD_POD_ID")) or Path("/workspace").is_dir()

# repo root : 노트북을 notebooks/ 에서 열든 repo root 에서 열든 eda/split_validation.py 가 있는 폴더를 찾음
REPO_ROOT = next((d for d in [Path.cwd(), *Path.cwd().parents] if (d / "eda" / "split_validation.py").exists()), Path.cwd())
ZIP_PATH  = Path(os.environ.get("STV_DATA_ZIP",  "/workspace/stv/data.zip" if ON_RUNPOD else str(REPO_ROOT / "data.zip")))
DATA_ROOT = Path(os.environ.get("STV_DATA_ROOT", "/root/data" if ON_RUNPOD else str(REPO_ROOT / "data")))
RAW_BASE_DIR   = DATA_ROOT / "raw"
SPLIT_BASE_DIR = DATA_ROOT / "split"
RAW_BASE_DIR.mkdir(parents=True, exist_ok=True)

# 1) data.zip → data/raw (train.csv 가 이미 있으면 skip)
train_csv_candidates = list(RAW_BASE_DIR.rglob("train.csv"))
if train_csv_candidates:
    print(f"[extract] skip: train.csv already exists → {train_csv_candidates[0]}")
else:
    if not ZIP_PATH.exists():
        raise FileNotFoundError(f"dataset zip not found: {ZIP_PATH}")
    print(f"[extract] {ZIP_PATH} → {RAW_BASE_DIR}")
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(RAW_BASE_DIR)

# zip 내부에 data/ 같은 최상위 폴더가 있어도 train.csv 위치를 기준으로 실제 raw root 탐색
train_csv_candidates = list(RAW_BASE_DIR.rglob("train.csv"))
if not train_csv_candidates:
    raise FileNotFoundError(f"train.csv not found under {RAW_BASE_DIR}")
if len(train_csv_candidates) > 1:
    print("[extract] multiple train.csv found; using:", train_csv_candidates[0])
RAW_DATA_ROOT = train_csv_candidates[0].parent
print("[data root]", RAW_DATA_ROOT)

# 2) train.csv → data/split/{train,validation}.csv (없으면 분할, 있으면 skip)
sys.path.insert(0, str(REPO_ROOT / "eda"))
from split_validation import ensure_split
split_paths = ensure_split(RAW_DATA_ROOT, SPLIT_BASE_DIR)
print("[split]", {k: str(v) for k, v in split_paths.items()})


# 설정

이 셀만 바꿔서 실험합니다. wandb 는 pod 환경변수 `WANDB_API_KEY`가 있으면 자동으로 켜집니다(`.env` → `runpod.py up`이 전달). **환경변수는 unsloth / transformers 를 import 하기 전에** 설정해야 적용됩니다.

In [ ]:
import os, time

# ── 환경 : 반드시 unsloth / transformers import "전"에 설정 ──
ON_RUNPOD = bool(os.environ.get("RUNPOD_POD_ID")) or os.path.isdir("/workspace")
if ON_RUNPOD:
    # 모델 캐시를 Network Volume에 → pod를 지워도 다음 up 에서 다시 받지 않음 (scripts/env.sh 와 같은 경로)
    os.environ.setdefault("HF_HOME", "/workspace/.cache/huggingface")
os.environ["UNSLOTH_DISABLE_STATISTICS"] = "1"   # Unsloth 사용 통계 전송 끄기
os.environ["TOKENIZERS_PARALLELISM"] = "false"
STV_USER = os.environ.get("STV_USER", "local")

# ── 경로 ──
ASSET_DIR  = "downloads"                          # (선택) 미리 받아 둔 모델 폴더: downloads/models/<이름>
from pathlib import Path
_repo_root    = next((d for d in [Path.cwd(), *Path.cwd().parents] if (d / "eda" / "split_validation.py").exists()), Path.cwd())
DATA_ROOT_DIR  = os.environ.get("STV_DATA_ROOT", "/root/data" if ON_RUNPOD else str(_repo_root / "data"))   # 전처리 셀과 동일
RAW_BASE_DIR   = os.path.join(DATA_ROOT_DIR, "raw")      # 원본 (이미지는 여기서 그대로 읽음)
SPLIT_DATA_DIR = os.path.join(DATA_ROOT_DIR, "split")    # train.csv / validation.csv (전처리 셀이 만듦)

# zip 내부 최상위 폴더 구조가 달라도 train.csv 위치를 기준으로 실제 raw root를 자동 탐색
_raw_train_csv = list(Path(RAW_BASE_DIR).rglob("train.csv"))
if not _raw_train_csv:
    raise FileNotFoundError(f"train.csv not found under {RAW_BASE_DIR}. 맨 앞 전처리 셀을 먼저 실행하세요.")
RAW_DATA_DIR = str(_raw_train_csv[0].parent)
if not os.path.exists(os.path.join(SPLIT_DATA_DIR, "validation.csv")):
    raise FileNotFoundError(f"validation.csv not found under {SPLIT_DATA_DIR}. 맨 앞 전처리 셀을 먼저 실행하세요.")
# 결과(어댑터·확률·제출 파일·로그)는 RunPod 에서 volume 에 → pod 를 지워도 남고 내 PC 에서 `bash scripts/s3.sh pull` 로 받음
OUTPUT_DIR = os.environ.get("STV_OUTPUT_DIR", f"/workspace/stv/outputs/{STV_USER}" if ON_RUNPOD else "output")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── 사전학습 모델 : 이미지를 입력받는 VL(Vision-Language) 모델이어야 합니다 ──
MODEL_NAME = "unsloth/Qwen3-VL-4B-Instruct-unsloth-bnb-4bit"     # Unsloth 가 미리 4bit 양자화한 모델
# MODEL_NAME = "unsloth/Qwen3-VL-8B-Instruct-unsloth-bnb-4bit"   # 더 크고 정확하지만 학습·추론 약 2배 느림 (사전 다운로드 필요)

def resolve_model_dir(name):
    """로컬 폴더 경로면 그대로, downloads/models/<폴더> 가 있으면 그 폴더, 그 외에는 HF 캐시(HF_HOME).
    캐시에 없으면 내려받음 (RunPod 에서는 volume 에 저장되어 처음 한 번만)"""
    if os.path.isdir(name):
        return name
    local = os.path.join(ASSET_DIR, "models", name.split("/")[-1])
    if os.path.isdir(local) and os.listdir(local):
        return local
    from huggingface_hub import snapshot_download
    try:
        return snapshot_download(name, local_files_only=True)
    except Exception:
        print(f"[model] 캐시에 없어 내려받습니다: {name} → {os.environ.get('HF_HOME', '~/.cache/huggingface')}")
        return snapshot_download(name, token=os.environ.get("HF_TOKEN") or None)

MODEL_DIR = resolve_model_dir(MODEL_NAME)

# ── 실험 설정 ──
SEED = 42

# 해상도 : 동적 (dynamic) + 종횡비 유지. 원본을 그대로 넣고 processor 가 픽셀 예산 안에서 28의 배수로 맞춤 (Qwen-VL smart_resize)
PATCH_PIXELS     = 28 * 28                 # 비전 토큰 1개 = 28×28 픽셀 (patch 14 × merge 2)
MIN_PIXELS       = 256 * PATCH_PIXELS      # 약 0.20MP. 이보다 작은 이미지는 여기까지 확대
TRAIN_MAX_PIXELS = 1280 * PATCH_PIXELS     # 약 1.00MP. 원본 중앙값 720×960(0.69MP)은 축소 없이 그대로 들어감 (EDA: 대부분 0.5~1MP)
INFER_MAX_PIXELS = TRAIN_MAX_PIXELS        # 추론 해상도 예산 (학습과 같게. 더 키우면 INFER_TAG 가 바뀜)
MAX_SEQ_LEN      = 2048                    # 이미지 토큰(최대 1280) + 텍스트 토큰 상한

# 데이터 : 검증은 data/split/validation.csv 고정 분할 (train의 10%, 이미지 해시 그룹 단위 + 정답 층화, seed 42 — eda/split_validation.py)
#          모든 실험이 같은 검증 문항으로 비교·앙상블됨. 분할을 바꾸려면 노트북이 아니라 스크립트를 수정하고 --force 로 다시 만들기
N_TRAIN         = None    # None = split/train.csv 전체. 빠른 실험은 1000 등
N_VALID         = None    # None = split/validation.csv 전체 (671문항). QUICK_RUN 에서만 줄임
USE_DEV         = False   # dev.csv(5명 응답) 다수결 라벨을 학습에 추가할지. 라벨 노이즈가 커서 기본 False (아래 '데이터 준비' 설명)
DEV_MIN_VOTES   = 3       # 5명 중 3명 이상이 같은 답을 고른 문항만 사용
SHUFFLE_OPTIONS = True    # 학습 때 보기 순서를 매번 무작위로 섞기 (정답 글자도 함께 변경)

# LoRA
LORA_R, LORA_ALPHA = 16, 16
FINETUNE_VISION    = True    # 비전 인코더도 LoRA 학습
FINETUNE_ATTENTION = True    # Attention 중심 LoRA
FINETUNE_MLP       = False   # MLP LoRA는 끔 → 속도/메모리 절약

# 학습
# 추천 실험 후보
#   LR:    2e-5 / 5e-5 / 1e-4
#   epoch: 0.5 / 1.0 / 1.5
EPOCHS      = 1.0
LR          = 1e-4        # 추천 후보: 2e-5 / 5e-5 / 1e-4
BATCH_SIZE  = 8
GRAD_ACCUM  = 1             # 유효 배치 = BATCH_SIZE x GRAD_ACCUM = 8
EVAL_STEPS  = 300           # optimizer step 마다 검증 정확도 측정 → 최고점 어댑터 저장 (1회 약 3분)
DO_TRAIN    = True          # False = 같은 RUN_NAME으로 저장된 best LoRA를 불러와 추론만 수행

# loss : 정답 글자 토큰만 남긴 뒤 표준 label smoothing 적용
LABEL_SMOOTHING = 0.10      # 0.0 = 기존 CE, 0.1 = 과확신 완화 baseline

# 프롬프트 : 학습·추론이 같은 프롬프트를 씀 ("a" 기본, "b" 글자 읽기 강조, "c" 한국어 시스템 프롬프트)
PROMPT_VARIANT = "a"
RUN_PROMPT_AB  = False      # True = 학습 전 모델로 세 프롬프트 점수 비교 (검증 300문항 x 3, 약 6분)

# 추론
N_TTA            = 2        # 보기 순서를 바꿔 여러 번 예측해 평균 (1 = 끔, 2 = 약 2배 시간, 4 = 약 4배)
INFER_BATCH_SIZE = 4

# 빠른 동작 확인 모드 : 전체 흐름이 에러 없이 도는지 몇 분 안에 확인
QUICK_RUN = os.environ.get("VQA_QUICK", "0") == "1"      # 직접 쓸 때는 True/False 로 바꿔도 됨
N_TEST = None
if QUICK_RUN:
    N_TRAIN, N_VALID, EVAL_STEPS, N_TEST = 48, 24, 3, 40

# 실험 이름 : 학습에 영향을 주는 설정값으로 자동 생성 (읽기 쉬운 앞부분 + 전체 학습 설정의 해시 6자리)
#   → 학습 설정을 바꾸면 어댑터·로그가 따로 저장됨 (앙상블에 활용)
#   → 추론 설정(N_TTA, INFER_MAX_PIXELS)은 INFER_TAG 로 확률·제출 파일 이름에만 붙음 (DO_TRAIN=False 로 추론만 다시 가능)
import hashlib
train_cfg = dict(model=MODEL_NAME, r=LORA_R, alpha=LORA_ALPHA, vision=FINETUNE_VISION,
                 attention=FINETUNE_ATTENTION, mlp=FINETUNE_MLP,
                 lr=LR, epochs=EPOCHS, seed=SEED,
                 batch=BATCH_SIZE, accum=GRAD_ACCUM, max_pixels=TRAIN_MAX_PIXELS, min_pixels=MIN_PIXELS,
                 label_smoothing=LABEL_SMOOTHING, n_train=N_TRAIN,
                 valid=N_VALID, split="data/split", eval_steps=EVAL_STEPS, dev=USE_DEV, dev_votes=DEV_MIN_VOTES,
                 shuffle=SHUFFLE_OPTIONS, prompt=PROMPT_VARIANT)
cfg_hash  = hashlib.md5(repr(sorted(train_cfg.items())).encode()).hexdigest()[:6]
MODEL_TAG = MODEL_NAME.split("/")[-1].split("-Instruct")[0].replace("-", "").lower()      # 예: qwen3vl4b
RUN_NAME  = f"{MODEL_TAG}_r{LORA_R}_lr{LR:g}_ep{EPOCHS}_s{SEED}_{cfg_hash}" + ("_quick" if QUICK_RUN else "")
INFER_TAG = f"tta{N_TTA}_maxpx{INFER_MAX_PIXELS // PATCH_PIXELS}"     # 예: tta2_maxpx1280 (비전 토큰 상한)
print("RUN_NAME:", RUN_NAME, "| INFER_TAG:", INFER_TAG)

ADAPTER_DIR = os.path.join(OUTPUT_DIR, f"{RUN_NAME}_best")
LOG_PATH    = os.path.join(OUTPUT_DIR, f"{RUN_NAME}_log.txt")   # 검증 정확도·loss 기록 (커널이 죽어도 남음)

def write_log(msg):
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(f"{time.strftime('%H:%M:%S')} {msg}\n")

# ── wandb : WANDB_API_KEY 가 있으면 켜짐. 실패해도 실험은 계속 (기록만 없음) ──
USE_WANDB = bool(os.environ.get("WANDB_API_KEY")) and not QUICK_RUN
wandb_run = None
if USE_WANDB:
    try:
        import wandb
        wandb_run = wandb.init(
            project=os.environ.get("WANDB_PROJECT", "stv"),
            entity=os.environ.get("WANDB_ENTITY") or None,
            name=f"{STV_USER}/{RUN_NAME}",
            id="".join(ch if ch.isalnum() or ch in "-_" else "-" for ch in f"{STV_USER}-{RUN_NAME}"),   # 같은 설정으로 다시 돌리면 같은 run 에 이어 씀
            resume="allow",
            config={**train_cfg, "user": STV_USER, "n_tta": N_TTA, "infer_tag": INFER_TAG, "do_train": DO_TRAIN},
            tags=[STV_USER, "runpod" if ON_RUNPOD else "local", "baseline"],
            dir=OUTPUT_DIR,
        )
        print("wandb:", wandb_run.url)
    except Exception as e:                        # 키 오류·네트워크 등
        print("⚠ wandb 초기화 실패 → 기록 없이 진행:", e)
        wandb_run = None

def wlog(data, step=None):
    """wandb 가 켜져 있을 때만 기록"""
    if wandb_run is not None:
        try:
            wandb_run.log(data, step=step)
        except Exception as e:
            print("⚠ wandb.log 실패:", e)

# 사전 준비 폴더 확인
for name, path in [("모델", MODEL_DIR), ("raw 데이터", RAW_DATA_DIR), ("split 데이터", SPLIT_DATA_DIR)]:
    state = "OK" if os.path.isdir(path) and os.listdir(path) else "없음 → 사전 준비 필요"
    print(f"{name:<4s} {state:<6s} {os.path.abspath(path)}")
print("QUICK_RUN:", QUICK_RUN, "| RunPod:", ON_RUNPOD, "| OUTPUT_DIR:", os.path.abspath(OUTPUT_DIR), "| wandb:", wandb_run is not None)


# 라이브러리

⚠ `import unsloth` 는 **transformers / peft / trl 보다 먼저** 해야 Unsloth 최적화가 적용됩니다. (이미 transformers 를 import 한 커널이면 Restart 후 위에서부터 다시 실행)

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)

In [ ]:
import unsloth                                   # 반드시 가장 먼저
from unsloth import FastVisionModel
from unsloth.trainer import UnslothVisionDataCollator

import re, math, random, time
import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset
from tqdm.auto import tqdm
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback
from peft import get_peft_model_state_dict, set_peft_model_state_dict

print(torch.__version__, "| CUDA:", torch.cuda.is_available())
assert torch.cuda.is_available(), "GPU 를 찾지 못했습니다. CUDA 버전 torch 가 설치된 커널인지 확인하세요."
print(torch.cuda.get_device_name(), "| bf16 지원:", torch.cuda.is_bf16_supported())

# Ampere 이상 NVIDIA GPU에서 matmul TF32 허용 (SFTConfig(tf32=True)와 동일한 의도)
torch.backends.cuda.matmul.allow_tf32 = True
if hasattr(torch.backends, "cudnn"):
    torch.backends.cudnn.allow_tf32 = True

Image.MAX_IMAGE_PIXELS = None     # 큰 이미지 로드 허용
CHOICES = ["a", "b", "c", "d"]
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)


# 데이터 준비

- **검증 데이터** : `data/split/validation.csv` (train 의 10% = 671문항) 를 그대로 씁니다. 매번 새로 나누지 않으므로 모든 실험·팀원이 같은 문항으로 비교됩니다. train 의 정답은 확실하므로 점수 비교 기준으로 씁니다.
- **학습 데이터** : `data/split/train.csv` (나머지 90%). 같은 이미지(해시 동일)가 검증에 있으면 학습에서 빠져 있습니다.
- **dev.csv** : 정답 대신 5명의 응답(answer1~5, 일부 공란)이 있습니다. 다수결 라벨을 만들 수 있지만 **기본값은 학습에 쓰지 않습니다(`USE_DEV=False`).** 직접 확인한 근거는 다음과 같습니다.
  - 4명 이상 합의한 문항이 **하나도 없고**(최다 득표가 3표 또는 2표), 응답자 각각은 다수결과 60% 정도만 일치합니다.
  - 파인튜닝 전 모델이 train 검증 문항은 약 92% 맞히는데, dev 다수결과는 약 63%만 일치합니다. 틀린 문항 이미지를 직접 보면 다수결이 틀린 경우가 많습니다. (예: `dev_0348` 간판에 "홀랄라 치킨 1층"이 적혀 있는데 다수결은 "시골집")
  - test 는 질문 길이·모델 확신도 분포가 dev 가 아니라 train 과 거의 같습니다.

In [ ]:
# 고정 분할 (전처리 셀이 만든 data/split). 이미지 path 는 raw 기준 상대 경로 그대로
train_df   = pd.read_csv(os.path.join(SPLIT_DATA_DIR, "train.csv"))
valid_df   = pd.read_csv(os.path.join(SPLIT_DATA_DIR, "validation.csv"))
test_df    = pd.read_csv(os.path.join(RAW_DATA_DIR, "test.csv"))
sample_sub = pd.read_csv(os.path.join(RAW_DATA_DIR, "sample_submission.csv"))
assert not set(train_df["id"]) & set(valid_df["id"]), "train/validation 에 같은 id 가 있음 → 분할 파일 점검"

if N_VALID:
    valid_df = valid_df.head(N_VALID).reset_index(drop=True)
if N_TRAIN:
    train_df = train_df.sample(n=min(N_TRAIN, len(train_df)), random_state=SEED)

def majority_vote(row):
    """answer1~5 다수결 → (라벨, 득표수). 공동 1위면 라벨 None"""
    votes = [str(row[f"answer{k}"]).strip().lower() for k in range(1, 6) if pd.notna(row[f"answer{k}"])]
    votes = [v for v in votes if v in CHOICES]
    if not votes:
        return None, 0
    counts = {c: votes.count(c) for c in CHOICES}
    top = max(counts.values())
    winners = [c for c in CHOICES if counts[c] == top]
    return (winners[0] if len(winners) == 1 else None), top

dev_df = None
dev_path = os.path.join(RAW_DATA_DIR, "dev.csv")
if os.path.exists(dev_path):
    dev_df = pd.read_csv(dev_path)
    mv = dev_df.apply(majority_vote, axis=1)
    dev_df["answer"] = [m[0] for m in mv]
    dev_df["votes"]  = [m[1] for m in mv]
    dev_hi = dev_df[(dev_df["votes"] >= DEV_MIN_VOTES) & dev_df["answer"].notna()]
    print(f"dev {len(dev_df)}문항 중 {DEV_MIN_VOTES}표 이상 합의 {len(dev_hi)}문항")
    if USE_DEV and not QUICK_RUN:
        train_df = pd.concat([train_df, dev_hi[train_df.columns]], ignore_index=True)

train_df = train_df.sample(frac=1, random_state=SEED).reset_index(drop=True)
if N_TEST:
    test_df = test_df.head(N_TEST)
print(f"train {len(train_df)} | valid {len(valid_df)} | test {len(test_df)}")
print("정답 분포(train):", train_df["answer"].value_counts().sort_index().to_dict())


# 모델 (Unsloth)

`FastVisionModel` 로 4bit base model을 불러오고 **새 Attention 중심 LoRA**를 붙입니다.

- `use_gradient_checkpointing="unsloth"` : Unsloth 방식 gradient checkpointing
- `finetune_vision_layers=True` : 비전 인코더도 LoRA 학습
- `finetune_attention_modules=True` : Attention 모듈 중심 학습
- `finetune_mlp_modules=False` : MLP LoRA 제외
- 기존 외부 adapter를 시작점으로 불러오는 로직은 사용하지 않습니다.

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - LoRA 구현 : LoraConfig()  →  Unsloth 에서는 FastVisionModel.get_peft_model()


In [ ]:
model, processor = FastVisionModel.from_pretrained(
    MODEL_DIR,
    load_in_4bit=True,                        # 4bit 양자화 (QLoRA)
    use_gradient_checkpointing="unsloth",
    max_seq_length=MAX_SEQ_LEN,
)

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=FINETUNE_VISION,
    finetune_language_layers=True,
    finetune_attention_modules=FINETUNE_ATTENTION,
    finetune_mlp_modules=FINETUNE_MLP,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias="none",
    random_state=SEED,
)
model.print_trainable_parameters()
tokenizer = processor.tokenizer

from safetensors.torch import load_file

def load_saved_lora(adapter_dir):
    """이 노트북에서 저장한 LoRA 가중치를 현재 모델에 로드합니다."""
    res = set_peft_model_state_dict(
        model,
        load_file(os.path.join(adapter_dir, "adapter_model.safetensors")),
    )
    missing_lora = [k for k in res.missing_keys if "lora_" in k]
    assert not res.unexpected_keys and not missing_lora, \
        f"저장된 LoRA 구조가 현재 설정과 다릅니다: {(res.unexpected_keys or missing_lora)[:3]}"

# fresh LoRA 초기 상태 저장:
# 학습 셀을 다시 실행해도 항상 동일한 초기점에서 시작합니다.
init_state = {
    k: v.detach().to("cpu", copy=True)
    for k, v in get_peft_model_state_dict(model).items()
}

print(f"GPU 메모리 사용: {torch.cuda.memory_allocated() / 2**30:.1f} GB")


# 프롬프트 템플릿, 해상도

학습과 추론은 같은 프롬프트 함수를 사용합니다.

이미지는 `load_image()`에서 **원본을 그대로** 읽습니다(리사이즈 없음). 크기 조정은 모델 processor 의 `smart_resize` 가 맡습니다.
**종횡비를 유지**한 채 총 픽셀이 `MIN_PIXELS`~`MAX_PIXELS` 안에 들어오도록 28의 배수로 반올림하므로, 원본이 예산 안이면 거의 원본 해상도로 들어갑니다.

> 아래 확인 코드에서 원본 크기 → 실제 encoder 입력 크기와 비전 토큰 수를 출력합니다. 원본 종횡비가 유지되는지 확인하세요.


In [ ]:
SYSTEM_EN = ("You are a helpful visual question answering assistant. "
             "Answer using exactly one letter among a, b, c, or d. No explanation.")
SYSTEM_KO = ("당신은 이미지를 보고 질문에 답하는 도우미입니다. "
             "a, b, c, d 중 정답 한 글자만 답하고 설명은 하지 마세요.")
ANSWER_RULE = "정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요."
PROMPTS = {                                   # 변형 : (시스템 프롬프트, 질문 뒤에 붙는 지시문)
    "a": (SYSTEM_EN, ANSWER_RULE),                                                  # baseline v3 와 동일
    "b": (SYSTEM_EN, "이미지 속 글자를 한 글자씩 주의 깊게 읽고 답하세요. " + ANSWER_RULE),
    "c": (SYSTEM_KO, ANSWER_RULE),
}

def build_mc_prompt(question, a, b, c, d, variant=None):
    return (
        f"{question}\n"
        f"(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n"
        + PROMPTS[variant or PROMPT_VARIANT][1]
    )

def build_messages(img, question, options, answer=None, variant=None):
    """options : 화면에 보여줄 순서의 보기 4개. answer 가 있으면 학습용(assistant 턴 포함)"""
    variant = variant or PROMPT_VARIANT
    messages = [
        {"role": "system", "content": [{"type": "text", "text": PROMPTS[variant][0]}]},
        {"role": "user", "content": [
            {"type": "image", "image": img},
            {"type": "text", "text": build_mc_prompt(question, *options)},
        ]},
    ]
    if answer is not None:
        messages.append({"role": "assistant", "content": [{"type": "text", "text": answer}]})
    return messages

def set_image_budget(max_pixels=TRAIN_MAX_PIXELS, min_pixels=MIN_PIXELS):
    # Qwen-VL processor 의 픽셀 예산 : 총 픽셀이 [min_pixels, max_pixels] 안에 들어오도록
    # 종횡비를 유지한 채 28의 배수로 리사이즈 (smart_resize). 원본이 예산 안이면 거의 그대로.
    processor.image_processor.size = {
        "shortest_edge": int(min_pixels),
        "longest_edge": int(max_pixels),
    }

def load_image(rel_path):
    path = os.path.join(RAW_DATA_DIR, rel_path)
    if not os.path.exists(path):
        raise FileNotFoundError(f"image missing: {path}. 맨 앞 전처리 셀을 다시 실행하세요.")
    return Image.open(path).convert("RGB")     # 원본 그대로. 크기 조정은 processor 가 담당

# 해상도 확인 : 원본 → encoder 입력 크기 (종횡비 유지 여부, 비전 토큰 수)
set_image_budget()
for p in list(train_df["path"].head(3)):
    img = load_image(p)
    grid = processor.image_processor(images=[img], return_tensors="pt")["image_grid_thw"][0]
    h = int(grid[1]) * processor.image_processor.patch_size
    w = int(grid[2]) * processor.image_processor.patch_size
    print(
        f"원본 {img.size[0]}x{img.size[1]} (비율 {img.size[0] / img.size[1]:.2f}) → processor 입력 {w}x{h} (비율 {w / h:.2f}) "
        f"| 비전 토큰 {int(grid.prod()) // processor.image_processor.merge_size**2}개 (상한 {TRAIN_MAX_PIXELS // PATCH_PIXELS})"
    )


# Custom Dataset, Collator — 정답 글자 1개만 loss

`train_on_responses_only=True`로 먼저 user/system/image 토큰을 마스킹한 뒤,
assistant 응답에서도 **정답 글자 `a/b/c/d`만 남기고 종료 토큰·줄바꿈은 다시 마스킹**합니다.

최종 loss:

```text
answer token only
    ↓
Cross Entropy + label smoothing(ε=0.1)
```

이번 baseline은 구현 안정성을 위해 Hugging Face `label_smoothing_factor`를 사용합니다.
즉 smoothing mass는 vocab 전체에 분배됩니다. **4지선다 전용 smoothing은 다음 실험으로 분리**합니다.


In [ ]:
class VQATrainDataset(Dataset):
    def __init__(self, df, shuffle_options=True, seed=SEED):
        self.df = df.reset_index(drop=True)
        self.shuffle_options = shuffle_options
        self.seed = seed
        self.n_seen = {}      # 같은 문항도 epoch 마다 다른 보기 순서가 되도록

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        options = [str(row[c]) for c in CHOICES]
        gold = CHOICES.index(str(row["answer"]).strip().lower())

        order = list(range(4))                               # order[j] = j번째 자리에 보여줄 원래 보기
        if self.shuffle_options:
            k = self.n_seen.get(i, 0); self.n_seen[i] = k + 1
            random.Random(self.seed * 1_000_003 + i * 7_919 + k).shuffle(order)
        shown = [options[j] for j in order]
        answer = CHOICES[order.index(gold)]

        img = load_image(row["path"])
        return {"messages": build_messages(img, str(row["question"]), shown, answer=answer)}

train_ds = VQATrainDataset(train_df, shuffle_options=SHUFFLE_OPTIONS)

CHOICE_IDS = [tokenizer.convert_tokens_to_ids(c) for c in CHOICES]
assert all(tokenizer.decode([t]) == c for t, c in zip(CHOICE_IDS, CHOICES)), "a/b/c/d가 단일 토큰인지 확인 필요"

_base_collator = UnslothVisionDataCollator(
    model, processor,
    resize="max",
    max_seq_length=MAX_SEQ_LEN,
    train_on_responses_only=True,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

def collator(features):
    batch = _base_collator(features)

    # assistant 응답 중 a/b/c/d 정답 글자만 loss 대상으로 유지
    labels = batch["labels"]
    keep = torch.zeros_like(labels, dtype=torch.bool)
    for token_id in CHOICE_IDS:
        keep |= labels.eq(token_id)

    batch["labels"] = torch.where(
        keep,
        labels,
        torch.full_like(labels, -100),
    )
    return batch

# 라벨 마스킹 검증: 샘플마다 정답 글자 1개만 남아야 함
set_image_budget()
check = collator([train_ds[0], train_ds[1]])
for ids, labels in zip(check["input_ids"], check["labels"]):
    kept = ids[labels != -100]
    print(f"시퀀스 길이 {len(ids):4d} | loss 대상 → {tokenizer.decode(kept)!r}")
    assert len(kept) == 1 and tokenizer.decode(kept) in CHOICES, "정답 글자 외 토큰이 loss에 포함됨"

# 평가 함수 (a/b/c/d 확률 직접 비교 + 보기 순서 TTA)

`generate` 로 글자를 생성한 뒤 파싱하지 않고, 답을 쓰기 직전 위치에서 **`a`,`b`,`c`,`d` 토큰의 확률만** 비교합니다. 파싱 실패가 없고 forward 1회로 끝납니다.

**보기 순서 TTA** : 보기를 순환 이동한 프롬프트로도 예측해서 원래 자리로 되돌려 평균합니다. 모델이 특정 글자(예: a)를 선호하는 위치 편향을 줄입니다.

#### 실습 참고 내용

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 추론 : with torch.no_grad(), model.eval()

In [ ]:
@torch.inference_mode()
def predict_probs(df, max_pixels=INFER_MAX_PIXELS, n_tta=1, batch_size=INFER_BATCH_SIZE, desc="predict",
                  cache_path=None, cache_key="", variant=None):
    """df 의 각 문항에 대해 원래 보기 a~d 의 확률 (N, 4) 반환. variant : 프롬프트 변형 (None = PROMPT_VARIANT)
    cache_path 를 주면 중간 결과를 저장 → 끊겨도 같은 설정(cache_key·TTA·해상도·문항)이면 이어서 실행"""
    variant = variant or PROMPT_VARIANT
    shifts = [0, 2, 1, 3][:n_tta]              # 보기 순환 이동량 (2 = 순서를 반 바퀴 돌림)
    probs = np.zeros((len(df), 4))
    done = np.zeros(len(df), dtype=bool)
    key = (f"{cache_key}|prompt{variant}|tta{n_tta}|px{max_pixels}-{MIN_PIXELS}|n{len(df)}"
           f"|{df['id'].iloc[0]}|{df['id'].iloc[-1]}")
    if cache_path and os.path.exists(cache_path):
        with np.load(cache_path) as saved:     # with : 파일을 바로 닫음 (Windows 에서 나중에 삭제 가능하도록)
            ok = "key" in saved.files and str(saved["key"]) == key
            if ok:
                probs, done = saved["probs"], saved["done"]
        print(f"[{desc}] " + (f"저장된 진행분 {done.sum()}/{len(df)} 이어서 실행" if ok else "다른 모델/설정의 캐시라서 무시하고 처음부터 실행"))

    old_size, old_side, was_training = dict(processor.image_processor.size), tokenizer.padding_side, model.training
    set_image_budget(max_pixels)
    tokenizer.padding_side = "left"            # 배치 추론 : 왼쪽 패딩이어야 마지막 위치가 답 자리
    model.eval()
    try:
        todo = np.where(~done)[0]
        for n_batch, start in enumerate(tqdm(range(0, len(todo), batch_size), desc=desc)):
            idx = todo[start:start + batch_size]
            rows = [df.iloc[i] for i in idx]
            imgs = [load_image(r["path"]) for r in rows]
            acc = np.zeros((len(idx), 4))
            for s in shifts:
                order = [(j + s) % 4 for j in range(4)]    # order[j] = j번째 자리에 보여줄 원래 보기
                texts = [
                    processor.apply_chat_template(
                        build_messages(img, str(r["question"]), [str(r[CHOICES[o]]) for o in order], variant=variant),
                        tokenize=False, add_generation_prompt=True)
                    for r, img in zip(rows, imgs)
                ]
                enc = processor(text=texts, images=imgs, padding=True, return_tensors="pt").to(model.device)
                with torch.autocast("cuda", dtype=torch.bfloat16):
                    logits = model(**enc, logits_to_keep=1).logits[:, -1, CHOICE_IDS]
                p = torch.softmax(logits.float(), dim=-1).cpu().numpy()
                acc[:, order] += p                         # j번째 자리의 확률 → 원래 보기 order[j] 자리로 누적
            probs[idx] = acc / len(shifts)
            done[idx] = True
            if cache_path and (n_batch % 50 == 49 or start + batch_size >= len(todo)):
                np.savez(cache_path, probs=probs, done=done, key=np.array(key))
    finally:                                   # 중간에 멈춰도 학습용 설정으로 되돌림
        processor.image_processor.size = old_size
        tokenizer.padding_side = old_side
        if was_training:
            model.train()
    return probs

def evaluate(df, probs, name="valid", show_wrong=5):
    gold = df["answer"].str.strip().str.lower().map(CHOICES.index).values
    pred = probs.argmax(1)
    correct = pred == gold
    numeric = df[CHOICES].astype(str).apply(lambda r: all(re.fullmatch(r"[\d\s,.\-:/%원₩$+()]+", x) for x in r), axis=1).values
    print(f"[{name}] 정확도 {correct.mean():.4f} ({correct.sum()}/{len(df)})"
          f" | 숫자형 보기 {correct[numeric].mean():.3f} (n={numeric.sum()})"
          f" | 텍스트형 보기 {correct[~numeric].mean():.3f} (n={(~numeric).sum()})")
    print("   예측 분포:", dict(zip(CHOICES, np.bincount(pred, minlength=4).tolist())))
    if show_wrong:
        wrong = df[~correct].head(show_wrong)
        for (_, r), pr in zip(wrong.iterrows(), pred[~correct][:show_wrong]):
            print(f"   ✗ {r['id']} | {r['question'][:40]} | 예측 {CHOICES[pr]}={str(r[CHOICES[pr]])[:15]} / 정답 {r['answer']}={str(r[r['answer']])[:15]}")
    return correct.mean()

# 시작 모델 점수 (step 0 기준선)

학습 전 **base model + fresh LoRA 초기 상태**의 검증 점수입니다.
LoRA는 초기화 시 원본 모델 출력을 유지하므로 사실상 zero-shot 기준선으로 사용합니다.

먼저 배치 추론(왼쪽 패딩)이 한 장씩 추론한 결과와 같은 답을 내는지 확인합니다.


In [ ]:
# 배치 추론 검증 : 한 장씩(batch 1) vs 배치(INFER_BATCH_SIZE) 예측이 같은 답인지 (bf16·패딩 때문에 확률은 약간 다를 수 있음)
chk = valid_df.head(8)
p1, pb = predict_probs(chk, batch_size=1, desc="batch=1"), predict_probs(chk, desc=f"batch={INFER_BATCH_SIZE}")
print(f"배치 vs 단일 : 답 일치 {(p1.argmax(1) == pb.argmax(1)).mean():.0%}, 확률 최대 차이 {np.abs(p1 - pb).max():.3f}")

t0 = time.time()
start_probs = predict_probs(valid_df, n_tta=1, desc="step 0 valid")
start_acc = evaluate(valid_df, start_probs, name="시작 모델 valid")
sec_per_sample = (time.time() - t0) / len(valid_df)
write_log(f"step 0 zero-shot valid_acc {start_acc:.4f}")
wlog({"valid/acc": start_acc, "valid/best_acc": start_acc}, step=0)
print(f"추론 속도 {sec_per_sample:.2f}초/문항 (TTA 없음). test 추론 시간은 '최종 검증' 셀에서 TTA 포함 속도로 다시 추정합니다")

# (선택) 프롬프트 A/B 테스트

`RUN_PROMPT_AB = True` 이면 세 가지 프롬프트의 점수를 같은 검증 문항으로 비교합니다.
학습 전 base model에서 비교하고, 가장 좋은 변형을 설정 셀의 `PROMPT_VARIANT` 에 넣은 뒤
**모델 셀부터 다시 실행**하세요. 학습·추론은 동일한 프롬프트를 사용합니다.


In [ ]:
if RUN_PROMPT_AB or QUICK_RUN:                 # QUICK_RUN 에서는 코드가 도는지만 확인
    ab_df = valid_df.head(300)
    for v in PROMPTS:
        evaluate(ab_df, predict_probs(ab_df, n_tta=1, desc=f"prompt {v}", variant=v), name=f"prompt {v}", show_wrong=0)
else:
    print("건너뜀 (RUN_PROMPT_AB = False)")

# fine-tuning (Unsloth + TRL SFTTrainer)

- LoRA 1 epoch
- 유효 batch = `BATCH_SIZE × GRAD_ACCUM`
- loss 대상 = **정답 글자 1개**
- `label_smoothing_factor = LABEL_SMOOTHING`
- `EVAL_STEPS`마다 validation accuracy 측정
- 가장 높은 accuracy의 LoRA를 자동 저장/복원

`LABEL_SMOOTHING=0.0`으로 바꾸면 기존 CE와 바로 비교할 수 있습니다.
공정 비교 시 나머지 seed/LR/split/pixel 설정은 그대로 유지하세요.


In [ ]:
class BestValidCallback(TrainerCallback):
    """EVAL_STEPS 마다 검증 정확도 측정 → 최고점이면 LoRA 가중치 저장.
    학습 전 상태(zero-shot step 0 점수)를 step 0 후보로 먼저 저장해 둠"""
    def __init__(self, model, start_acc):
        self.history = [(0, start_acc)]
        self._save_best(model, start_acc, 0)

    def _save_best(self, model, acc, step):
        self.best_acc, self.best_step = acc, step
        self.best_state = {k: v.detach().to("cpu", copy=True) for k, v in get_peft_model_state_dict(model).items()}
        model.save_pretrained(ADAPTER_DIR)

    def _eval(self, state, model):
        probs = predict_probs(valid_df, n_tta=1, desc=f"valid@{state.global_step}")
        acc = evaluate(valid_df, probs, name=f"step {state.global_step}", show_wrong=0)
        self.history.append((state.global_step, acc))
        write_log(f"step {state.global_step}/{state.max_steps} valid_acc {acc:.4f}")
        if acc > self.best_acc:
            self._save_best(model, acc, state.global_step)
            print(f"   ★ best 갱신 → {ADAPTER_DIR}")
        wlog({"valid/acc": acc, "valid/best_acc": self.best_acc}, step=state.global_step)

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs:
            write_log(f"step {state.global_step}/{state.max_steps} loss {logs['loss']:.4f} lr {logs.get('learning_rate', 0):.2e}")
            wlog({"train/loss": logs["loss"], "train/learning_rate": logs.get("learning_rate", 0),
                  "train/grad_norm": logs.get("grad_norm", 0), "train/epoch": logs.get("epoch", 0)}, step=state.global_step)

    def on_step_end(self, args, state, control, model=None, **kwargs):
        if state.global_step % EVAL_STEPS == 0 or state.global_step == state.max_steps:
            self._eval(state, model)

if DO_TRAIN:
    set_peft_model_state_dict(model, init_state)   # 항상 시작점에서 출발 (이 셀을 다시 실행해도 step 0 점수와 일치)
    set_image_budget(TRAIN_MAX_PIXELS)
    FastVisionModel.for_training(model)
    best_cb = BestValidCallback(model, start_acc)

    trainer = SFTTrainer(
        model=model,
        processing_class=tokenizer,
        data_collator=collator,
        train_dataset=train_ds,
        callbacks=[best_cb],
        args=SFTConfig(
            output_dir=os.path.join(OUTPUT_DIR, "trainer"),
            per_device_train_batch_size=BATCH_SIZE,
            gradient_accumulation_steps=GRAD_ACCUM,
            num_train_epochs=EPOCHS,
            learning_rate=LR,
            lr_scheduler_type="cosine",
            warmup_ratio=0.05,
            weight_decay=0.01,
            optim="adamw_8bit",
            max_grad_norm=1.0,
            label_smoothing_factor=LABEL_SMOOTHING,
            bf16=True,
            tf32=True,
            logging_steps=10,
            save_strategy="no",             # 저장은 BestValidCallback 이 담당
            report_to="none",
            seed=SEED,
            # --- 비전 데이터용 필수 설정 (Unsloth 권장) ---
            remove_unused_columns=False,
            dataset_text_field="",
            dataset_kwargs={"skip_prepare_dataset": True},
            max_length=MAX_SEQ_LEN,
            dataloader_num_workers=4,
            dataloader_persistent_workers=True,
            dataloader_prefetch_factor=2,
        ),
    )
    t0 = time.time()
    try:
        train_result = trainer.train()
        print(f"학습 시간 {(time.time() - t0) / 60:.1f}분 | 평균 train loss {train_result.training_loss:.4f}")
    except KeyboardInterrupt:
        print("⚠ 학습을 중단했습니다 → 지금까지의 best 어댑터로 복원합니다")
    train_min = (time.time() - t0) / 60
    write_log(f"train end {train_min:.1f} min, best step {best_cb.best_step} valid_acc {best_cb.best_acc:.4f}")
    if wandb_run is not None:
        wandb_run.summary.update({"valid/start_acc": start_acc, "valid/best_step": best_cb.best_step,
                                  "valid/best_acc": best_cb.best_acc, "train_min": round(train_min, 1)})
    print("검증 정확도 기록 (step, 정확도):", [(s, round(float(a), 4)) for s, a in best_cb.history])

    # 최고점 어댑터로 되돌리기
    set_peft_model_state_dict(model, best_cb.best_state)
    print(f"best step {best_cb.best_step} (검증 {best_cb.best_acc:.4f}) 어댑터로 복원")
    if best_cb.best_step == 0:
        print("⚠ 어떤 checkpoint 도 zero-shot 기준보다 좋지 않아 zero-shot 초기 상태를 그대로 사용합니다."
              + (" (QUICK_RUN 은 검증 문항이 적어 정상일 수 있음)" if QUICK_RUN else " LR·데이터를 점검하세요."))
else:
    # 학습 없이 추론: 이 노트북에서 같은 설정으로 저장한 best LoRA만 사용
    train_min = 0.0
    adapter_path = os.path.join(ADAPTER_DIR, "adapter_model.safetensors")
    if not os.path.exists(adapter_path):
        raise FileNotFoundError(
            f"{ADAPTER_DIR} 에 학습된 LoRA가 없습니다. DO_TRAIN=True 로 먼저 학습하세요."
        )

    import json
    cfg = json.load(open(os.path.join(ADAPTER_DIR, "adapter_config.json"), encoding="utf-8"))
    assert (cfg["r"], cfg["lora_alpha"]) == (LORA_R, LORA_ALPHA), \
        f"저장된 LoRA(r={cfg['r']}, alpha={cfg['lora_alpha']})와 현재 설정이 다릅니다"

    load_saved_lora(ADAPTER_DIR)
    print("저장된 best LoRA 로드:", ADAPTER_DIR)

_ = processor.save_pretrained(ADAPTER_DIR)


# 최종 검증

최고점 어댑터로 검증 데이터를 TTA 와 함께 다시 평가합니다.

- 검증 확률 `output/probs_valid_{RUN_NAME}_{INFER_TAG}.npy` 저장 → 앙상블 조합을 검증 점수로 고를 때 사용
- 실험 결과 한 줄을 `output/experiments.csv` 에 누적 → 실험끼리 비교
- 모델이 **높은 확신(>0.95)으로 라벨과 다르게 답한 문항**을 `output/label_suspects_{RUN_NAME}.csv` 로 저장 → 라벨 오류 후보를 눈으로 확인

In [ ]:
FastVisionModel.for_inference(model)
t0 = time.time()
valid_probs = predict_probs(valid_df, n_tta=N_TTA, desc="valid (TTA)")
print(f"TTA{N_TTA} 추론 {(time.time() - t0) / len(valid_df):.2f}초/문항 → test {len(test_df)}문항 예상 약 {(time.time() - t0) / len(valid_df) * len(test_df) / 60:.0f}분")
final_acc = evaluate(valid_df, valid_probs, name=f"valid TTA{N_TTA}")

# 과확신 간단 점검
_pred = valid_probs.argmax(1)
_pmax = valid_probs.max(1)
_gold = valid_df["answer"].str.strip().str.lower().map(CHOICES.index).values
_wrong = _pred != _gold
print(
    f"confidence audit | p>=0.99: {(_pmax >= 0.99).sum()}/{len(valid_df)} "
    f"| wrong & p>=0.90: {(_wrong & (_pmax >= 0.90)).sum()}/{_wrong.sum() if _wrong.sum() else 0}"
)
write_log(f"final valid_acc TTA{N_TTA} {final_acc:.4f}")
if wandb_run is not None:
    wandb_run.summary.update({f"valid/final_acc_tta{N_TTA}": final_acc})
if DO_TRAIN:
    print(f"시작 모델 {start_acc:.4f} → 파인튜닝(best step {best_cb.best_step}, TTA 없음) {best_cb.best_acc:.4f}"
          f" → + TTA{N_TTA} {final_acc:.4f}")
else:
    print(f"시작 모델(TTA 없음) {start_acc:.4f} | 사용한 어댑터 + TTA{N_TTA} {final_acc:.4f}")
print(f"※ 검증 {len(valid_df)}문항 기준 표준오차 약 ±{np.sqrt(final_acc * (1 - final_acc) / len(valid_df)) * 100:.1f}%p"
      " → 이보다 작은 차이는 우연일 수 있음")
np.save(os.path.join(OUTPUT_DIR, f"probs_valid_{RUN_NAME}_{INFER_TAG}.npy"), valid_probs)

# 실험 기록 누적
if not QUICK_RUN:
    exp_path = os.path.join(OUTPUT_DIR, "experiments.csv")
    row = {"time": time.strftime("%Y-%m-%d %H:%M"), "run": RUN_NAME, "infer": INFER_TAG, **train_cfg,
           "start": round(start_acc, 4), "best_step": best_cb.best_step if DO_TRAIN else None,
           "best_valid": round(best_cb.best_acc, 4) if DO_TRAIN else None, "final_valid": round(final_acc, 4),
           "train_min": round(train_min, 1)}
    pd.DataFrame([row]).to_csv(exp_path, mode="a", header=not os.path.exists(exp_path), index=False, encoding="utf-8-sig")
    print("실험 기록 →", exp_path)

# 라벨 오류 후보 : 모델이 높은 확신으로 라벨과 다르게 답한 검증 문항
pred, pmax = valid_probs.argmax(1), valid_probs.max(1)
gold = valid_df["answer"].str.strip().str.lower().map(CHOICES.index).values
suspect = (pmax > 0.95) & (pred != gold)
sus_df = valid_df.loc[suspect, ["id", "path", "question", *CHOICES, "answer"]].assign(
    pred=[CHOICES[i] for i in pred[suspect]], prob=pmax[suspect].round(3))
sus_df.to_csv(os.path.join(OUTPUT_DIR, f"label_suspects_{RUN_NAME}.csv"), index=False, encoding="utf-8-sig")
print(f"라벨 오류 후보 {len(sus_df)}문항 → output/label_suspects_{RUN_NAME}.csv (이미지를 열어 직접 확인해 보세요)")

# inference (test) + 제출 파일

- 제출 파일 : **`OUTPUT_DIR/submission.csv`** (RunPod: `/workspace/stv/outputs/{STV_USER}/`, 내 PC에서 `bash scripts/s3.sh pull`) (실험별 사본 `output/submission_{RUN_NAME}_{INFER_TAG}.csv`)
- 문항별 a~d 확률을 `output/probs_test_{RUN_NAME}_{INFER_TAG}.npy` 로 저장합니다. 여러 실험의 확률을 평균하면 앙상블이 됩니다.
- 중간에 끊겨도 같은 셀을 다시 실행하면 이어서 예측합니다. (같은 어댑터·TTA·해상도일 때만. 다 끝나면 이어하기 캐시는 삭제)

#### 실습 참고 내용

    챕터4-1 RAG 기반 Customer Service AI 에이전트 개발
    - 데이터 파서 : 문자열 파싱 대신 토큰 확률로 선택지 결정

In [ ]:
cache_path = os.path.join(OUTPUT_DIR, f"cache_test_{RUN_NAME}.npz")
adapter_id = os.path.getmtime(os.path.join(ADAPTER_DIR, "adapter_model.safetensors"))   # 어댑터가 바뀌면 캐시 무효
test_probs = predict_probs(test_df, n_tta=N_TTA, desc=f"test (TTA{N_TTA})",
                           cache_path=cache_path, cache_key=f"{RUN_NAME}|{adapter_id}")
np.save(os.path.join(OUTPUT_DIR, f"probs_test_{RUN_NAME}_{INFER_TAG}.npy"), test_probs)
if os.path.exists(cache_path):
    os.remove(cache_path)                      # 끝까지 예측했으면 이어하기용 캐시는 삭제

submission = pd.DataFrame({"id": test_df["id"].values, "answer": [CHOICES[i] for i in test_probs.argmax(1)]})

# 제출 전 점검
assert submission["answer"].isin(CHOICES).all()
if N_TEST is None:
    assert len(submission) == len(sample_sub) and (submission["id"].values == sample_sub["id"].values).all(), "id 순서/개수가 sample_submission 과 다름"
dist = submission["answer"].value_counts(normalize=True).sort_index()
print("예측 분포:", dist.round(3).to_dict())
assert dist.max() < 0.5, "한 글자로 예측이 쏠림 → 모델/프롬프트 점검 필요"

SUBMISSION_PATH = os.path.join(OUTPUT_DIR, f"submission_{RUN_NAME}_{INFER_TAG}.csv")
submission.to_csv(SUBMISSION_PATH, index=False)
if N_TEST is None:                             # QUICK_RUN(일부 문항)은 제출용 파일을 덮어쓰지 않음
    submission.to_csv(os.path.join(OUTPUT_DIR, "submission.csv"), index=False)
print("Saved", SUBMISSION_PATH, "" if N_TEST else "+ output/submission.csv")
write_log(f"saved {SUBMISSION_PATH}")
if wandb_run is not None:
    wandb_run.summary.update({"test/n": len(submission), "test/pred_dist": dist.round(3).to_dict()})
    wandb_run.finish()                           # 앙상블 셀은 wandb 에 기록하지 않음
submission.head()

# 다음 실험 순서

| 실험 | 변경값 |
|---|---|
| 기존 CE 재현 | `LABEL_SMOOTHING = 0.0` |
| 기본 smoothing | `LABEL_SMOOTHING = 0.1` |
| 강한 smoothing 확인 | `LABEL_SMOOTHING = 0.2` |
| 다음 단계 | a/b/c/d 4개에만 smoothing하는 custom loss |
| 마지막 후보 | hard negative를 고른 DPO |

비교할 때는 **해상도 예산(MIN/MAX_PIXELS), 검증 분할(data/split), LR, epoch, LoRA 설정을 모두 고정**하고 loss만 바꿉니다.

OOM이면 해상도 예산은 바꾸지 말고 먼저 `BATCH_SIZE=1`, `GRAD_ACCUM=8`로 바꿔 유효 batch 8을 유지하세요.


In [ ]:
# 앙상블 : 저장해 둔 실험들의 확률 평균. 실험(RUN_NAME)마다 가장 최근 추론 결과 1개만 사용
import glob
latest = {}
for f in sorted(glob.glob(os.path.join(OUTPUT_DIR, "probs_test_*.npy")), key=os.path.getmtime):
    if "_quick" in f:
        continue
    run = os.path.basename(f)[len("probs_test_"):].rsplit("_tta", 1)[0]
    valid_f = f.replace("probs_test_", "probs_valid_")
    if os.path.exists(valid_f):                # 검증 확률이 있는 실험만 (같은 data/split 검증 분할이어야 비교 가능)
        latest[run] = (f, valid_f)

if len(latest) >= 2 and not QUICK_RUN:
    runs = [r for r in latest if np.load(latest[r][1]).shape == (len(valid_df), 4)]
    for r in runs:
        evaluate(valid_df, np.load(latest[r][1]), name=r, show_wrong=0)
    ens_valid = np.mean([np.load(latest[r][1]) for r in runs], axis=0)
    evaluate(valid_df, ens_valid, name=f"앙상블 {len(runs)}개", show_wrong=0)
    ens = np.mean([np.load(latest[r][0]) for r in runs], axis=0)
    ens_sub = pd.DataFrame({"id": test_df["id"].values, "answer": [CHOICES[i] for i in ens.argmax(1)]})
    ens_sub.to_csv(os.path.join(OUTPUT_DIR, "submission_ensemble.csv"), index=False)
    print("Saved output/submission_ensemble.csv  (앙상블 검증 점수가 단일 실험보다 높을 때만 제출하세요)")
else:
    print(f"앙상블할 실험이 {len(latest)}개 → 2개 이상 필요 (SEED·LR·모델을 바꿔 한 번 더 실행)")